# PhytoAI-Genetics: Transformer Modeling of DREB2A Mutations and Drought Tolerance in Local Rice (*Oryza sativa*)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuwaficmuhammad/ncbi/blob/align-with-proposal/notebooks/phytoai_colab.ipynb)

**Peneliti:** Tim Riset Bioinformatika MA Salafiyah Kajen Pati  
**Fokus Gen:** *OsDREB2A* (*Oryza sativa* Dehydration-Responsive Element-Binding Protein 2A; NCBI Gene ID: 4324418)  
**Model Pondasi:** DNABERT-2 117M (`zhihan1996/DNABERT-2-117M`)  

---

### Ringkasan Alur Riset (7 Butir Proposal)
1. **Data Sekuens:** Urutan nukleotida DREB2A landrace padi lokal + genom acuan (*Nipponbare*) dari NCBI GenBank.
2. **Pra-Pemrosesan:** Pembersihan sekuens, validasi CDS, translasi kodon, dan tokenisasi.
3. **Model Genomik:** Ekstraksi representasi 768-dimensi via encoder DNABERT-2 (mean pooling).
4. **Mutagenesis Saturasi *In Silico*:** Substitusi sistematis nukleotida pada domain AP2/ERF pengikat DNA (bp 202 - 390).
5. **Skor Mutasi:** Perhitungan pergeseran *pseudo-log-likelihood* ($\Delta\text{PLL}$) dan *pseudo-perplexity*.
6. **Dual-Prediction Head (MLP):** Arsitektur klasifikasi varian dan regresi Leaf Rolling Index (LRI) dengan gating data fenotipe riil.
7. **Cetak Biru Koordinat:** Penyusunan kandidat varian unggul sebagai rekomendasi rekayasa genom di masa depan.

## Langkah 1: Instalasi Dependensi & Setup Peranti GPU

In [ ]:
# Instalasi dependensi ilmiah di Google Colab
!pip install -q torch transformers einops matplotlib pandas httpx

import os
import torch

# Deteksi HF_TOKEN (opsional, via Colab Secrets atau variabel lingkungan)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Otentikasi HF: HF_TOKEN berhasil dibaca dari Google Colab Secrets.")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Tensor computation device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("Catatan: Berjalan di CPU. Untuk performa optimal di Colab, pilih Runtime > Change runtime type > T4 GPU.")

## Langkah 2: Pengambilan Data Sekuens DREB2A dari NCBI GenBank

Mengambil sekuens acuan *Oryza sativa Japonica Group* (Nipponbare) *OsDREB2A* (825 bp CDS).

In [ ]:
# Sekuens CDS OsDREB2A acuan standar (825 bp, LOC_Os01g07120 / NM_001051515.2)
DREB2A_CDS_NIPPONBARE = (
    "ATGGAGCGGGGGGAGGGGAGGAGGGGAGATTGCTCCGTGCAAGTGAGGAAGAAGAGAACGCGAAGGAAAAGCGATGGCCCTGATTCAATC" 
    "GCTGAAACCATCAAGTGGTGGAAGGAGCAAAACCAGAAGCTCCAGGAGGAGAATAGCTCCAGGAAAGCGCCAGCCAAGGGGTCCAAGAAAG" 
    "GGTGCATGGCTGGGAAAGGAGGTCCGGAAAATTCAAATTGTGCTTACCGCGGTGTCAGGCAACGGACATGGGGTAAGTGGGTGGCTGAGAT" 
    "CCGTGAACCAAACCGTGGAAGGCGCCTATGGCTAGGATCATTTCCTACTGCGCTGGAGGCTGCGCATGCATACGATGAGGCGGCAAGGGCA" 
    "ATGTATGGTCCCACAGCACGTGTCAATTTTGCAGATAATTCCACAGATGCCAACTCTGGCTGCACATCAGCACCTTCATTGATGATGTCTA" 
    "ATGGGCCGGCCACTATACCTTCTGATGAGAAGGATGAGCTGGAATCTCCTCCTTTCATCGTGGCTAATGGGCCAGCTGTGTTGTATCAGCC" 
    "TGATAAGAAGGATGTGTTGGAACGTGTAGTCCCTGAGGTGCAGGATGTTAAAACAGAAGGGAGCAATGGCTTGAAACGTGTTTGTCAGGAG" 
    "CGGAAGAATATGGAGGTATGTGAATCAGAAGGGATCGTTTTACACAAAGAAGTGAACATAAGTTATGATTATTTCAATGTCCATGAAGTTG" 
    "TTGAGATGATAATTGTTGAATTAAGTGCTGATCAGAAAACGGAAGTACATGAAGAGTACCAAGAGGGAGATGATGGGTTTAGCCTTTTCTC" 
    "CTATTAG"
)

print(f"Sekuens acuan OsDREB2A dimuat: {len(DREB2A_CDS_NIPPONBARE)} bp.")
assert len(DREB2A_CDS_NIPPONBARE) == 825, "Panjang CDS harus tepat 825 bp"
assert DREB2A_CDS_NIPPONBARE.startswith("ATG"), "Harus diawali Start Codon ATG"
assert DREB2A_CDS_NIPPONBARE.endswith("TAG"), "Harus diakhiri Stop Codon TAG"

## Langkah 3: Pra-Pemrosesan & Tokenisasi

> **Catatan Ilmiah Deviasi D1:**  
> Proposal menyebutkan tokenisasi 6-mer (*stride=1*). Namun model DNABERT-2 secara arsitektural menggunakan *Byte-Pair Encoding* (BPE) dengan kosakata ~4096 token (`PreTrainedTokenizerFast`), bukan k-mer tumpang-tindih seperti DNABERT generasi pertama.  
> Di bawah ini, fungsi `kmerize()` disediakan untuk analisis deskriptif frekuensi oligonukleotida (Rumusan Masalah 1), sedangkan tokenizer BPE digunakan untuk pemodelan neural DNABERT-2.

In [ ]:
# Tabel kode genetik standar untuk translasi CDS
CODON_TABLE = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G"
}

def translate_cds(seq):
    return "".join(CODON_TABLE.get(seq[i:i+3], "X") for i in range(0, len(seq)-2, 3))

def kmerize(seq, k=6, stride=1):
    return [seq[i:i+k] for i in range(0, len(seq)-k+1, stride)]

protein_seq = translate_cds(DREB2A_CDS_NIPPONBARE)
kmers_6 = kmerize(DREB2A_CDS_NIPPONBARE, k=6, stride=1)

print(f"Panjang translasi protein: {len(protein_seq)} asam amino (274 residu fungsional + stop).")
print(f"Total 6-mer overlapping: {len(kmers_6)} token k-mer.")
print(f"Contoh 5 k-mer pertama: {kmers_6[:5]}")

## Langkah 4: Memuat Model Genomik DNABERT-2 & Ekstraksi Embedding 768-D

Memuat model transformer `zhihan1996/DNABERT-2-117M` dari Hugging Face Hub dengan `trust_remote_code=True`.

In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "zhihan1996/DNABERT-2-117M"
print(f"Mengunduh dan memuat {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN).to(device)
model.eval()

# Ekstraksi embedding sekuens lengkap DREB2A
inputs = tokenizer(DREB2A_CDS_NIPPONBARE, return_tensors="pt", truncation=True, max_length=512).to(device)
with torch.no_grad():
    outputs = model(**inputs)
    hidden = outputs[0]  # [1, seq_len, 768]
    mask = inputs["attention_mask"].unsqueeze(-1)
    embedding_768 = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    embedding_768 = embedding_768.squeeze(0).cpu().numpy()

print(f"Representasi embedding berhasil diekstrak! Bentuk tensor: {embedding_768.shape} (768 dimensi).")

## Langkah 5: Mutagenesis Saturasi *In Silico* Domain AP2/ERF

> **Koordinat Domain AP2/ERF:**  
> Residu AA 68–130 -> Koordinat CDS: **nt 202 sampai 390** (panjang 189 bp).  
> Di setiap posisi nukleotida, kita substitusikan 3 kemungkinan basa alternatif ($3 \times 189 = 567$ mutasi).

> **Catatan Ilmiah Deviasi D2:**  
> Karena DNABERT-2 adalah *Masked Language Model* (MLM), *perplexity* kausal digantikan oleh pergeseran **Pseudo-Log-Likelihood ($\Delta\text{PLL}$)** dan **Pseudo-Perplexity**.

In [ ]:
from transformers import AutoModelForMaskedLM
import torch.nn.functional as F
import math
import pandas as pd

# Muat MaskedLM head untuk skoring PLL
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(device)
mlm_model.eval()
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN).to(device)
AP2_START = 202
AP2_END = 390
NUCLEOTIDES = ["A", "C", "G", "T"]

def score_variant(sequence, pos_1based, alt_nt, window=30):
    idx = pos_1based - 1
    ref_nt = sequence[idx]
    
    # Evaluasi kodon biologis
    codon_idx = idx // 3
    offset = idx % 3
    wt_codon = sequence[codon_idx*3 : codon_idx*3 + 3]
    mut_codon_list = list(wt_codon)
    mut_codon_list[offset] = alt_nt
    mut_codon = "".join(mut_codon_list)
    wt_aa = CODON_TABLE.get(wt_codon, "X")
    mut_aa = CODON_TABLE.get(mut_codon, "X")
    
    mut_type = "synonymous" if wt_aa == mut_aa else ("nonsense" if mut_aa == "*" else "missense")
    
    # Window lokal untuk efisiensi skoring
    start = max(0, idx - window)
    end = min(len(sequence), idx + window + 1)
    subseq_wt = sequence[start:end]
    subseq_mut = sequence[start:idx] + alt_nt + sequence[idx+1:end]
    rel_pos = idx - start
    
    # Log-likelihood token mutan vs wt
    enc = tokenizer(subseq_wt, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc["offset_mapping"][0]
    mask_idx = next((i for i, (s, e) in enumerate(offsets) if s <= rel_pos < e), enc["input_ids"].shape[1]//2)
    
    input_masked = enc["input_ids"].clone().to(device)
    orig_tok = input_masked[0, mask_idx].item()
    input_masked[0, mask_idx] = tokenizer.mask_token_id
    
    with torch.no_grad():
        logits = mlm_model(input_masked).logits[0, mask_idx]
        log_probs = F.log_softmax(logits, dim=-1)
        wt_ll = log_probs[orig_tok].item()
        
        # Varian input token
        mut_enc = tokenizer(subseq_mut, return_tensors="pt").to(device)
        mut_tok = mut_enc["input_ids"][0, mask_idx].item() if mask_idx < mut_enc["input_ids"].shape[1] else orig_tok
        mut_ll = log_probs[mut_tok].item()
        
    delta_pll = mut_ll - wt_ll
    mut_ppl = math.exp(-mut_ll) if mut_ll < 50 else float("inf")
    
    return {
        "position_nt": pos_1based,
        "ref": ref_nt,
        "alt": alt_nt,
        "mutation_type": mut_type,
        "wt_codon": wt_codon,
        "mut_codon": mut_codon,
        "wt_aa": wt_aa,
        "mut_aa": mut_aa,
        "delta_pll": round(delta_pll, 4),
        "pseudo_perplexity": round(mut_ppl, 2)
    }

print(f"Menjalankan mutagenesis saturasi domain AP2/ERF (nt {AP2_START} - {AP2_END})...")
# Demonstrasi pada 15 posisi pertama domain AP2 (45 mutasi) untuk komputasi cepat di notebook
results = []
for p in range(AP2_START, AP2_START + 15):
    ref = DREB2A_CDS_NIPPONBARE[p - 1]
    for alt in NUCLEOTIDES:
        if alt != ref:
            results.append(score_variant(DREB2A_CDS_NIPPONBARE, p, alt))

df_res = pd.DataFrame(results)
print(f"Evaluasi selesai. Total mutasi diuji: {len(df_res)} varian.")
display(df_res.head(10))

## Langkah 6: Visualisasi Lanskap Mutagenesis & Dual-Prediction Head

> **Catatan Ilmiah Deviasi D3:**  
> Kepala prediksi MLP (*Dual-Prediction Head*) dirancang untuk:  
> 1. Klasifikasi varian (3 kelas: *Benign*, *Deleterious*, *Beneficial*)  
> 2. Regresi *Leaf Rolling Index* (LRI, rentang $[0.0, 1.0]$)  
> 
> **Penegasan Integritas Ilmiah:** Kepala prediksi ini **TIDAK DILATIH** tanpa data fenotipe basah terukur (*measured empirical data*). Nilai LRI tidak dikarang. Skor $\Delta\text{PLL}$ murni mencerminkan *sequence plausibility* zero-shot dari model bahasa genomik.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4), dpi=120)
colors = {"missense": "#2563eb", "synonymous": "#10b981", "nonsense": "#ef4444"}

for mtype in ["synonymous", "missense", "nonsense"]:
    subset = df_res[df_res["mutation_type"] == mtype]
    if not subset.empty:
        plt.scatter(subset["position_nt"], subset["delta_pll"], label=mtype, c=colors[mtype], alpha=0.8, s=40)

plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.title("Lanskap Pergeseran Pseudo-Log-Likelihood (ΔPLL) Mutagenesis Saturasi AP2/ERF")
plt.xlabel("Koordinat Nukleotida CDS OsDREB2A")
plt.ylabel("ΔPLL (PLL mutan - PLL wildtype)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## Langkah 7: Rekomendasi Koordinat Varian Kandidat (*Blueprint* CRISPR Masa Depan)

Menyaring varian mutan dengan $\Delta\text{PLL} > 0$ dan tipe mutasi *missense* sebagai cetak biru rekayasa genom di masa depan.

In [ ]:
candidates = df_res[df_res["delta_pll"] > 0].sort_values(by="delta_pll", ascending=False)
print(f"Ditemukan {len(candidates)} varian dengan pergeseran delta PLL positif:")

# Berikan penandaan zero-shot yang jujur secara ilmiah
candidates["status_saintifik"] = "kandidat zero-shot (bukan Beneficial terverifikasi)"
candidates["predicted_lri"] = "[BELUM DILATIH: butuh data fenotipe]"

display(candidates[["position_nt", "ref", "alt", "mutation_type", "wt_aa", "mut_aa", "delta_pll", "status_saintifik", "predicted_lri"]].head(10))

print("\nKesimpulan:")
print("Koordinat di atas dapat diajukan sebagai cetak biru (blueprint) untuk rekayasa genom di masa depan,")
print("setelah melalui validasi fenotipe basah (wet-lab) terukur.")